# Day 2 Lab：让职业数字人开始有流程

在开始之前，我们先快速复习一下今天学了什么。

---

## 今天我们先讲清了一件事

Anthropic 把 AI 智能体系统分成两种：

- **workflow**
- **agent**

**workflow** 是你先把路设计好，AI 按路走。  
**agent** 是 AI 自己决定下一步要做什么。

这门课，我们先从 **workflow** 开始。

因为它更：

- 稳
- 清楚
- 可控

刚开始搭系统的时候，  
先把路搭稳，比一开始追求很高的自主性更重要。

---

## 今天我们还看了 5 种常见的 workflow pattern

### 1. Prompt Chaining｜提示链
一步接一步。  
前一步的结果，交给下一步。

### 2. Routing｜路由分流
先判断这是什么任务，  
再走不同的路。

### 3. Parallelization｜并行处理
几个分支同时做，  
最后再把结果合起来。

### 4. Evaluator-Optimizer｜评估—优化
先做一版，  
再检查一版，  
再继续优化。

### 5. Orchestrator-Workers｜总控—执行者
先看整个任务，  
再动态拆成几块，  
分给不同执行者去做。

---

## 今天最重要的感觉

这些模式不是拿来背的。  
它们更像是几种组织工作的方式。

真实系统里，  
往往会把几种模式组合起来用。

但流程也不是拆得越细越好。  
只有当后面的处理方式真的不同，  
这个分流才值得做。

---

## 那今天这份 notebook 要做什么？

今天，我们不加工具，  
也不加资料库。

我们先做一个最小的 **Routing system**。

也就是让职业数字人先学会：

**判断你现在要它干什么，  
再决定走哪条路。**

今天它先支持这几类任务：

- 自我介绍类
- 写作辅助类
- 数据分析类
- 工作安排 / 总结类
- 不明确问题先追问

---

## 今天这份 notebook 的目标

做完以后，你会得到：

**职业数字人 v1：会分流任务的助手**

它现在还不会真的分析表格，  
也还不会真的去读资料。

但它已经会先判断：

这个问题以后该往哪条路走。

---

好，接下来我们开始动手。

## 先做准备工作：把今天要用的大脑接上

在真正开始做 routing 之前，  
我们先把今天要用的大模型接上。

这里先做两件事：

- 导入今天会用到的工具
- 连接模型，准备一个最小调用函数

这样后面做任务分类和分流的时候，  
代码会更清楚，也更容易复用。

In [1]:
# 导入工具
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown

# 读取 .env 里的 API key
load_dotenv()

# 创建客户端（通义千问兼容 OpenAI 接口）
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("API_KEY")
)

def call_llm(user_question: str, system_prompt: str) -> str:
    response = client.chat.completions.create(
        model="qwen3.6-plus",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

display(Markdown("### ✅ 模型已连接"))

### ✅ 模型已连接

## 读取基础资料：CV 和工作日志

接下来，我们把职业数字人最基础的两份资料读进来：

- `CV.txt`
- `work_log.txt`

这两份资料会先作为今天 routing system 的背景信息。

它们现在还不是知识库。  
今天我们只是先把它们读进来，  
让职业数字人在回答自我介绍类、工作相关问题时，  
有最基本的上下文。

In [2]:
# 读取简历和工作日志
with open("data/CV.txt", "r", encoding="utf-8") as f:
    resume = f.read()

with open("data/work_log.txt", "r", encoding="utf-8") as f:
    work_log = f.read()

display(Markdown("### ✅ 资料读取完成"))

### ✅ 资料读取完成

## 第一步：先把今天要分的几类任务说清楚

在开始写 routing 之前，  
我们先把今天的任务类别定下来。

因为一个最小的 routing system，  
最重要的第一步不是写很多代码，  
而是先知道：

**我们到底要分哪几类。**

今天，我们先支持这 5 类任务：

- `profile`：自我介绍类
- `writing`：写作辅助类
- `data_analysis`：数据分析类
- `work_planning`：工作安排 / 总结类
- `clarify`：不明确问题先追问

接下来，我们会先把这些类别定义出来。

In [3]:
# 定义任务类别
task_types = [
    "profile",
    "writing",
    "data_analysis",
    "work_planning",
    "clarify"
]

display(Markdown("### 今天的任务类别"))
for task in task_types:
    display(Markdown(f"- `{task}`"))

### 今天的任务类别

- `profile`

- `writing`

- `data_analysis`

- `work_planning`

- `clarify`

## 第二步：让模型先判断这是什么任务

现在，我们先不急着让职业数字人回答很多内容。

我们先让它做一件更基础、也更重要的事：

**先判断这句话属于哪一类任务。**

这就是 routing 的第一步。

In [4]:
router_system_prompt = f"""
你是一个任务分类器。

你的工作不是回答用户问题，
而是先判断这个问题属于哪一类任务。

可选任务类别只有这 5 个：
- profile
- writing
- data_analysis
- work_planning
- clarify

分类规则：
- profile：用户在问你是谁、你做什么、你做过什么项目
- writing：用户要你写、改、润色、总结一段文字
- data_analysis：用户要你分析表格、数据、结果
- work_planning：用户要你整理待办、总结工作、安排下一步
- clarify：用户的问题太模糊，暂时无法直接判断，需要先追问

你只能返回上面 5 个标签中的一个。
不要解释，不要输出别的内容。
"""

## 第三步：先试一句话，看看它会怎么分

接下来，我们先用一句很简单的话试一下。

看看模型会不会先把任务类型判断对。

In [5]:
# 测试一下这个分类器：
test_question = "帮我写一段周报开头"

task_label = call_llm(
    user_question=test_question,
    system_prompt=router_system_prompt
)

display(Markdown("### 测试输入"))
display(Markdown(test_question))

display(Markdown("### 模型判断的任务类别"))
display(Markdown(f"`{task_label}`"))

### 测试输入

帮我写一段周报开头

### 模型判断的任务类别

`writing`

## 第四步：根据分类结果，决定下一步怎么走

现在，模型已经先判断出了任务类型。

接下来，我们先不急着让职业数字人把所有事情做完。

我们先让它决定：

- 这是什么任务
- 下一步应该做什么
- 需不需要先向用户要更多信息

这样后面我们就能很自然地把 Prompt Chaining 接上去。

In [7]:
def get_next_step(task_label: str, user_question: str) -> dict:
    if task_label == "profile":
        return {
            "task_label": "profile",
            "next_step": "answer_profile",
            "needs_more_info": False,
            "message": "这是自我介绍类任务，可以直接进入简短回答。"
        }

    elif task_label == "writing":
        return {
            "task_label": "writing",
            "next_step": "clarify_writing_requirements",
            "needs_more_info": True,
            "message": "这是写作辅助类任务，先确认语气、场景和要求。"
        }

    elif task_label == "data_analysis":
        return {
            "task_label": "data_analysis",
            "next_step": "ask_for_data",
            "needs_more_info": True,
            "message": "这是数据分析类任务，先向用户要表格、数据或指标。"
        }

    elif task_label == "work_planning":
        return {
            "task_label": "work_planning",
            "next_step": "ask_for_todo_or_context",
            "needs_more_info": True,
            "message": "这是工作安排 / 总结类任务，先向用户要待办、背景或时间安排。"
        }

    else:
        return {
            "task_label": "clarify",
            "next_step": "ask_clarifying_question",
            "needs_more_info": True,
            "message": "这个问题还不够明确，先追问。"
        }


route_result = get_next_step(task_label, test_question)

display(Markdown("### 分流结果"))
display(Markdown(f"- 任务类别：`{route_result['task_label']}`"))
display(Markdown(f"- 下一步：`{route_result['next_step']}`"))
display(Markdown(f"- 是否需要补充信息：`{route_result['needs_more_info']}`"))
display(Markdown(f"- 说明：{route_result['message']}"))

### 分流结果

- 任务类别：`writing`

- 下一步：`clarify_writing_requirements`

- 是否需要补充信息：`True`

- 说明：这是写作辅助类任务，先确认语气、场景和要求。

## 到这里，我们的智能体已经知道怎么分流了

现在，职业数字人已经不会把所有问题都当成同一种任务来处理了。

它会先判断：

- 这是自我介绍类问题
- 还是写作辅助类问题
- 还是数据分析类问题
- 还是工作安排 / 总结类问题
- 或者，这个问题还需要先追问

也就是说，  
它已经开始有了第一层系统感：

**先判断任务，  
再决定往哪走。**

这一步很重要。

因为后面不管是接工具、接资料，还是加入更多上下文，  
都要先有这个分流框架，系统才会更清楚。

## 第五步：根据分流结果，开始执行下一步

到这里，职业数字人已经知道这是什么任务了。

接下来，我们不再停在“判断类别”这一步。  
我们开始让它真正往前走一小步。

也就是：

- 如果是 `profile`，就先做一个简短回答
- 如果是 `writing`，就先确认写作要求
- 如果是 `data_analysis`，就先向用户要数据
- 如果是 `work_planning`，就先向用户要待办或背景
- 如果是 `clarify`，就先追问

这样，职业数字人就不只是“会分流”，  
而是已经开始知道：

**分流之后，自己下一步该做什么。**

In [11]:
def run_next_step(route_result: dict, user_question: str, resume: str, work_log: str) -> str:
    task_label = route_result["task_label"]
    next_step = route_result["next_step"]

    if next_step == "answer_profile":
        system_prompt = f"""
        你是一个职业数字人。

        请根据下面的简历和工作日志信息，
        用积极、友好、简洁的口吻回答用户的问题。

        要求：
        - 控制在 2 到 4 句
        - 不要太长
        - 不要编造资料里没有的内容

        【简历】
        {resume}

        【工作日志】
        {work_log}
        """
        return call_llm(user_question=user_question, system_prompt=system_prompt)

    elif next_step == "clarify_writing_requirements":
        system_prompt = """
        你是一个职业数字人，正在帮助用户完成写作任务。

        你的目标是先判断：用户现在提供的信息，是否已经足够开始写。

        如果信息还不够，请你用自然、友好、简洁的口吻，追问最关键的缺失信息。
        优先确认这些内容：
        - 使用场景
        - 语气风格
        - 受众对象
        - 长度要求
        - 是否有必须包含的信息

        如果用户已经说得比较清楚了，就不要继续追问，直接开始起草第一版。

        要求：
        - 语气自然，不要机械
        - 不要一次问太多，优先问最关键的 1 到 2 个问题
        - 如果已经可以开写，就直接给出草稿
        """
        return call_llm(user_question=user_question, system_prompt=system_prompt)
    

    elif next_step == "ask_for_data":
        return "可以。把数据表发给我，或者直接把数据贴过来。你也可以顺手告诉我，你最想看哪些指标。"

    elif next_step == "ask_for_todo_or_context":
        return "可以。先把你现在的待办、时间安排，或者这周最重要的几件事告诉我，我再帮你一起理顺。"

    else:
        return "我先确认一下，你现在最想让我帮你做的，是写东西、分析数据，还是整理工作安排？"
        

final_response = run_next_step(route_result, test_question, resume, work_log)

display(Markdown("### 职业数字人的下一步动作"))
display(Markdown(final_response))

### 职业数字人的下一步动作

没问题！为了更贴合你的实际工作，想先跟你确认两个小细节：
1. 这段周报主要是给谁看的？（比如直属领导、团队内部，还是跨部门/高层）
2. 你希望整体语气偏正式严谨，还是轻松务实、偏结果导向？

告诉我后我马上为你起草第一版～

试着把剩下几条通路的下一步也设计出来。

不用追求一步到位。  
先让每一条路都能往前走一小步，就已经很好了。


In [14]:
test_question = "帮我分析一下这个表格，看看我这个月的工作效率怎么样？"
final_response = run_next_step(route_result, test_question, resume, work_log)

display(Markdown("### 职业数字人的下一步动作"))
display(Markdown(final_response))

### 职业数字人的下一步动作

没问题，我很乐意帮你分析！不过你好像还没把表格发过来～可以先发给我看看吗？

另外想顺便确认一下：这份分析主要是给你自己复盘用，还是需要整理成汇报给领导/团队的版本？这样我能更好地把握语气和侧重点。

## 今天先到这里

很好。  
到这里，你已经让职业数字人有了第一层真正的系统感。

它不再是一个“什么问题都直接回答”的模型了。  
它已经开始会先判断任务，再决定往哪一条路走。

今天，我们先把 **routing** 和最小的 **prompt chaining** 搭起来。  
后面你会看到：

- Day 3，我们会把工具接上去
- Day 4，我们会把资料和上下文接上去

也就是说，今天这一步，已经是在给后面的能力打底了。

你不用追求一次写得很完美。  
能先把流程搭起来，已经很厉害了。

---

## Further Reading

如果你想继续往下看，可以回顾这些材料：

- Anthropic：**Building Effective Agents**
- OpenAI：**Techniques to Improve Reliability**
- OpenAI：**Orchestration and Handoffs**

你现在不需要把它们全背下来。  
只要记住一句话就够了：

**先判断任务，再决定下一步。**

这就是 workflow 开始真正成立的地方。